# Luck Weight Calibration Analysis

## Empirical Validation of Composite Luck Score Weights

This notebook validates the weights used in the composite luck score formula:

**Current Formula** (from `fct_advanced_luck.sql`):
```sql
composite_luck_score = 50
    + (wins_over_expected) * 10           -- ±10 per win
    + (schedule_luck_index) * -0.5        -- Schedule harder = unlucky
    + (close_game_win_pct - 0.5) * 20     -- Close game variance
```

### Peer Review Feedback:
> "Using arbitrary weights (10, -0.5, 20) without empirical justification. Need variance decomposition to weight ∝ variance explained."

### Methodology:
1. **Variance Decomposition**: Regress `wins_over_expected` on schedule and close-game components
2. **Cross-Validation**: Test which weights best predict rest-of-season wins
3. **Sensitivity Analysis**: Show how composite score changes with ±20% weight adjustments

### Expected Outcome:
Either validate current weights OR recommend new calibrated weights based on data.

---

In [1]:
import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Connect to warehouse
conn = duckdb.connect('../data/warehouse.duckdb', read_only=True)
print("✅ Connected to warehouse")

✅ Connected to warehouse


## 1. Load Current Luck Data

Get the existing composite luck scores and their components.

In [2]:
# Load luck data with all components
luck_df = conn.execute("""
    SELECT 
        roster_id,
        manager_name,
        actual_wins,
        actual_losses,
        expected_wins,
        wins_over_expected,
        schedule_luck_index,
        close_wins,
        close_losses,
        total_close_games,
        close_game_win_pct,
        composite_luck_score,
        luck_rating
    FROM main_analytics.fct_advanced_luck
    ORDER BY composite_luck_score DESC
""").df()

print(f"Loaded {len(luck_df)} teams")
print(f"\nLuck Score Range: {luck_df['composite_luck_score'].min():.1f} - {luck_df['composite_luck_score'].max():.1f}")
print(f"Mean: {luck_df['composite_luck_score'].mean():.1f}, Std: {luck_df['composite_luck_score'].std():.1f}")

# Display summary
luck_df.head()

Loaded 12 teams

Luck Score Range: 29.8 - 74.2
Mean: 50.3, Std: 13.6


,roster_id,manager_name,actual_wins,actual_losses,expected_wins,wins_over_expected,schedule_luck_index,close_wins,close_losses,total_close_games,close_game_win_pct,composite_luck_score,luck_rating
0,12,georgeuhrick,3.0,3.0,2.18,0.82,-11.98,1.0,0.0,1.0,1.000,74.2,VERY LUCKY
1,11,mrdorsey,4.0,2.0,3.64,0.36,-7.58,1.0,0.0,1.0,1.000,67.4,VERY LUCKY
2,9,mrbeef1,5.0,1.0,4.09,0.91,-1.01,2.0,1.0,3.0,0.667,62.9,Lucky
3,6,AKMCG,4.0,2.0,3.55,0.45,-8.81,0.0,0.0,0.0,NaN,58.9,Lucky
4,4,jacklamb,4.0,2.0,3.27,0.73,1.88,0.0,0.0,0.0,NaN,56.4,Lucky


## 2. Variance Decomposition

**Question**: How much variance in `wins_over_expected` is explained by:
1. Schedule luck (opponent strength/timing)
2. Close game luck (coin-flip variance)

**Method**: Linear regression to get R² for each component.

In [3]:
# Prepare features for regression
# Handle missing close_game_win_pct (teams with 0 close games)
luck_analysis = luck_df.copy()
luck_analysis['close_game_deviation'] = luck_analysis['close_game_win_pct'].fillna(0.5) - 0.5

# Features
X = luck_analysis[['schedule_luck_index', 'close_game_deviation']].values
y = luck_analysis['wins_over_expected'].values

# Fit regression
model = LinearRegression()
model.fit(X, y)

# Predictions and metrics
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print("Variance Decomposition Results")
print("=" * 60)
print(f"\nR² (variance explained): {r2:.3f}")
print(f"RMSE: {rmse:.3f} wins")
print(f"\nRegression Coefficients:")
print(f"  Schedule Luck Index:    {model.coef_[0]:.4f}")
print(f"  Close Game Deviation:   {model.coef_[1]:.4f}")
print(f"  Intercept:              {model.intercept_:.4f}")

print(f"\n**Current Model Weights** (from SQL):")
print(f"  Schedule: -0.5")
print(f"  Close Game: 20.0")
print(f"\n**Data-Driven Weights** (from regression):")
print(f"  Schedule: {model.coef_[0]:.2f}")
print(f"  Close Game: {model.coef_[1]:.2f}")

Variance Decomposition Results

R² (variance explained): 0.464
RMSE: 0.515 wins

Regression Coefficients:
  Schedule Luck Index:    -0.0504
  Close Game Deviation:   0.6502
  Intercept:              -0.0090

**Current Model Weights** (from SQL):
  Schedule: -0.5
  Close Game: 20.0

**Data-Driven Weights** (from regression):
  Schedule: -0.05
  Close Game: 0.65


## 3. Visualize Actual vs Predicted Wins Over Expected

How well do schedule + close-game luck explain total luck?

In [4]:
luck_analysis['predicted_woe'] = y_pred
luck_analysis['residual'] = y - y_pred

fig = px.scatter(
    luck_analysis,
    x='predicted_woe',
    y='wins_over_expected',
    hover_data=['manager_name', 'actual_wins', 'expected_wins'],
    title=f'Actual vs Predicted Wins Over Expected (R² = {r2:.3f})',
    labels={
        'predicted_woe': 'Predicted WOE (Schedule + Close Games)',
        'wins_over_expected': 'Actual Wins Over Expected'
    }
)

# Add diagonal line (perfect prediction)
max_val = max(luck_analysis['wins_over_expected'].max(), luck_analysis['predicted_woe'].max())
min_val = min(luck_analysis['wins_over_expected'].min(), luck_analysis['predicted_woe'].min())

fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='gray', dash='dash'),
    name='Perfect Prediction',
    showlegend=True
))

fig.update_layout(height=500, template='plotly_white')
fig.show()

print("\nTeams with Largest Residuals (unexplained luck):")
print("=" * 60)
print(luck_analysis.nlargest(3, 'residual')[['manager_name', 'wins_over_expected', 'predicted_woe', 'residual']])


Teams with Largest Residuals (unexplained luck):
    manager_name  wins_over_expected  predicted_woe  residual
4       jacklamb                0.73      -0.103858  0.833858
2        mrbeef1                0.91       0.150475  0.759525
5  jamespancakes                0.09      -0.437707  0.527707


## 4. Sensitivity Analysis

Test how composite luck score changes with ±20% weight adjustments.

**Goal**: Verify scores are stable (small weight changes don't drastically re-rank teams).

In [5]:
# Test different weight configurations
weight_scenarios = {
    'Current (SQL)': {'woe': 10, 'schedule': -0.5, 'close': 20},
    'Data-Driven': {'woe': 10, 'schedule': model.coef_[0], 'close': model.coef_[1]},
    'Schedule +20%': {'woe': 10, 'schedule': -0.6, 'close': 20},
    'Schedule -20%': {'woe': 10, 'schedule': -0.4, 'close': 20},
    'Close +20%': {'woe': 10, 'schedule': -0.5, 'close': 24},
    'Close -20%': {'woe': 10, 'schedule': -0.5, 'close': 16},
}

def calc_composite_score(row, weights):
    """Recalculate composite luck score with custom weights"""
    return (
        50 
        + row['wins_over_expected'] * weights['woe']
        + row['schedule_luck_index'] * weights['schedule']
        + (row['close_game_win_pct'] if pd.notna(row['close_game_win_pct']) else 0.5 - 0.5) * weights['close']
    )

# Calculate scores for each scenario
for scenario_name, weights in weight_scenarios.items():
    luck_analysis[f'score_{scenario_name}'] = luck_analysis.apply(
        lambda row: calc_composite_score(row, weights), axis=1
    )

# Show comparison for top 5 managers
print("Composite Luck Score Sensitivity (Top 5 by Current Formula)")
print("=" * 80)
top_5 = luck_analysis.nlargest(5, 'composite_luck_score')

for idx, row in top_5.iterrows():
    print(f"\n{row['manager_name']} ({row['actual_wins']}-{row['actual_losses']}, WOE: {row['wins_over_expected']:.1f})")
    for scenario_name in weight_scenarios.keys():
        score = row[f'score_{scenario_name}']
        delta = score - row['composite_luck_score']
        print(f"  {scenario_name:20s}: {score:5.1f} (Δ {delta:+5.1f})")

Composite Luck Score Sensitivity (Top 5 by Current Formula)

georgeuhrick (3.0-3.0, WOE: 0.8)
  Current (SQL)       :  84.2 (Δ +10.0)
  Data-Driven         :  59.5 (Δ -14.7)
  Schedule +20%       :  85.4 (Δ +11.2)
  Schedule -20%       :  83.0 (Δ  +8.8)
  Close +20%          :  88.2 (Δ +14.0)
  Close -20%          :  80.2 (Δ  +6.0)

mrdorsey (4.0-2.0, WOE: 0.4)
  Current (SQL)       :  77.4 (Δ +10.0)
  Data-Driven         :  54.6 (Δ -12.8)
  Schedule +20%       :  78.1 (Δ +10.7)
  Schedule -20%       :  76.6 (Δ  +9.2)
  Close +20%          :  81.4 (Δ +14.0)
  Close -20%          :  73.4 (Δ  +6.0)

mrbeef1 (5.0-1.0, WOE: 0.9)
  Current (SQL)       :  72.9 (Δ +10.0)
  Data-Driven         :  59.6 (Δ  -3.3)
  Schedule +20%       :  73.0 (Δ +10.1)
  Schedule -20%       :  72.8 (Δ  +9.9)
  Close +20%          :  75.6 (Δ +12.7)
  Close -20%          :  70.3 (Δ  +7.4)

AKMCG (4.0-2.0, WOE: 0.5)
  Current (SQL)       :  58.9 (Δ  +0.0)
  Data-Driven         :  54.9 (Δ  -4.0)
  Schedule +20%     

## 5. Rank Correlation Test

**Stability Metric**: How much do team rankings change with different weights?

Use Spearman correlation to measure rank stability.

In [6]:
from scipy.stats import spearmanr

# Calculate ranks for each scenario
for scenario_name in weight_scenarios.keys():
    luck_analysis[f'rank_{scenario_name}'] = luck_analysis[f'score_{scenario_name}'].rank(ascending=False)

# Compute correlations with current formula
base_ranks = luck_analysis['rank_Current (SQL)'].values
correlations = {}

for scenario_name in weight_scenarios.keys():
    if scenario_name != 'Current (SQL)':
        scenario_ranks = luck_analysis[f'rank_{scenario_name}'].values
        corr, p_value = spearmanr(base_ranks, scenario_ranks)
        correlations[scenario_name] = corr

print("Rank Correlation with Current Formula")
print("=" * 60)
for scenario, corr in sorted(correlations.items(), key=lambda x: x[1], reverse=True):
    print(f"{scenario:20s}: ρ = {corr:.4f}")

print(f"\n✅ Good stability if ρ > 0.95 (ranks mostly unchanged)")
print(f"⚠️  Unstable if ρ < 0.90 (significant re-ranking)")

Rank Correlation with Current Formula
Close -20%          : ρ = 1.0000
Schedule +20%       : ρ = 0.9790
Schedule -20%       : ρ = 0.9790
Close +20%          : ρ = 0.9580
Data-Driven         : ρ = 0.9091

✅ Good stability if ρ > 0.95 (ranks mostly unchanged)
⚠️  Unstable if ρ < 0.90 (significant re-ranking)


## 6. Recommendations

Based on the variance decomposition and sensitivity analysis above.

In [7]:
print("=" * 80)
print("WEIGHT CALIBRATION RECOMMENDATIONS")
print("=" * 80)

print(f"\n1. **Variance Explained (R²)**: {r2:.1%}")
if r2 > 0.80:
    print("   ✅ EXCELLENT: Schedule + close games explain most luck variance")
elif r2 > 0.60:
    print("   ✓ GOOD: Majority of variance explained, some unexplained factors exist")
else:
    print("   ⚠️  WEAK: Significant unexplained variance - consider additional factors")

print(f"\n2. **Current Weights vs Data-Driven Weights**:")
print(f"   Current:      schedule = -0.5,  close_game = 20.0")
print(f"   Data-Driven:  schedule = {model.coef_[0]:.2f}, close_game = {model.coef_[1]:.2f}")

ratio_schedule = abs(model.coef_[0] / -0.5)
ratio_close = abs(model.coef_[1] / 20.0)

if 0.8 < ratio_schedule < 1.2 and 0.8 < ratio_close < 1.2:
    print("   ✅ VALIDATED: Current weights align well with data (within 20%)")
else:
    print("   ⚠️  MISALIGNED: Consider updating to data-driven weights")

print(f"\n3. **Sensitivity/Stability**:")
min_corr = min(correlations.values()) if correlations else 1.0
if min_corr > 0.95:
    print(f"   ✅ STABLE: Minimum ρ = {min_corr:.3f} - rankings robust to weight changes")
elif min_corr > 0.90:
    print(f"   ✓ ACCEPTABLE: Minimum ρ = {min_corr:.3f} - mostly stable")
else:
    print(f"   ⚠️  UNSTABLE: Minimum ρ = {min_corr:.3f} - rankings change significantly")

print(f"\n{'='*80}")
print("FINAL VERDICT")
print("=" * 80)

if r2 > 0.70 and 0.8 < ratio_schedule < 1.2 and 0.8 < ratio_close < 1.2:
    print("✅ **KEEP CURRENT WEIGHTS** - Empirically validated")
    print("   Current formula is well-calibrated. No changes needed.")
else:
    print("📊 **CONSIDER DATA-DRIVEN WEIGHTS** - Better empirical fit")
    print(f"   Suggested update in fct_advanced_luck.sql:")
    print(f"   - schedule_luck_index coefficient: {model.coef_[0]:.2f}")
    print(f"   - close_game deviation coefficient: {model.coef_[1]:.2f}")

WEIGHT CALIBRATION RECOMMENDATIONS

1. **Variance Explained (R²)**: 46.4%
   ⚠️  WEAK: Significant unexplained variance - consider additional factors

2. **Current Weights vs Data-Driven Weights**:
   Current:      schedule = -0.5,  close_game = 20.0
   Data-Driven:  schedule = -0.05, close_game = 0.65
   ⚠️  MISALIGNED: Consider updating to data-driven weights

3. **Sensitivity/Stability**:
   ✓ ACCEPTABLE: Minimum ρ = 0.909 - mostly stable

FINAL VERDICT
📊 **CONSIDER DATA-DRIVEN WEIGHTS** - Better empirical fit
   Suggested update in fct_advanced_luck.sql:
   - schedule_luck_index coefficient: -0.05
   - close_game deviation coefficient: 0.65


In [ ]:
# Luck Weight Calibration Analysis

## Empirical Validation of Composite Luck Score Weights

This notebook validates the weights used in the composite luck score formula:

**Current Formula** (from `fct_advanced_luck.sql`):
```sql
composite_luck_score = 50
    + (wins_over_expected) * 10           -- ±10 per win
    + (schedule_luck_index) * -0.5        -- Schedule harder = unlucky
    + (close_game_win_pct - 0.5) * 20     -- Close game variance
```

### Peer Review Feedback:
> "Using arbitrary weights (10, -0.5, 20) without empirical justification. Need variance decomposition to weight ∝ variance explained."

### Methodology:
1. **Variance Decomposition**: Regress `wins_over_expected` on schedule and close-game components
2. **Cross-Validation**: Test which weights best predict rest-of-season wins
3. **Sensitivity Analysis**: Show how composite score changes with ±20% weight adjustments

### Expected Outcome:
Either validate current weights OR recommend new calibrated weights based on data.

---

In [8]:
# Close connection to allow dbt to rebuild models
conn.close()
print("✅ Connection closed")

✅ Connection closed
